# 🤖 Fine-tune Chatbot ตารางสอนอาจารย์ ด้วย HuggingFace (LoRA)

Notebook นี้นำ Dataset ที่เตรียมไว้แล้ว (`schedule_chatbot_train.jsonl`, `schedule_chatbot_validation.jsonl`)
มา **Fine-tune** โมเดลภาษาด้วยเทคนิค **LoRA (Low-Rank Adaptation)** ผ่าน HuggingFace `transformers` +
`peft` + `trl`

ข้อมูลในโฟลเดอร์ `dataset/` ปัจจุบัน: **train 1,668 คู่ / validation 185 คู่** (ยังไม่มีไฟล์ test — ขั้นตอน
ทดสอบจะใช้ validation แทนอัตโนมัติ) จำนวนจริงจะถูกพิมพ์ออกมาตอนรัน Cell โหลด Dataset

## แนวทางที่เลือกใช้ และเหตุผล

- **Dataset ยังไม่ใหญ่ (~1.8 พันคู่) และเนื้อหาซ้ำแนวกันสูง** — การเทรนใหม่ทั้งหมด (from scratch) หรือ Full
  Fine-tuning จะ overfit ง่ายและไม่คุ้มทรัพยากร จึงเลือกใช้ **LoRA** ซึ่งเทรนเฉพาะพารามิเตอร์ส่วนน้อย
  (Adapter) บนโมเดลฐานที่ฉลาดอยู่แล้ว ลดความเสี่ยง Overfitting และใช้ GPU น้อยกว่ามาก
- **โมเดลฐาน (Base model)** เริ่มต้นใช้ `Qwen/Qwen2.5-1.5B-Instruct` ซึ่งรองรับภาษาไทยได้ในระดับดี ขนาดเล็ก
  พอจะรันบน GPU ฟรีของ Kaggle (T4, 16GB) ได้สบาย หากต้องการคุณภาพสูงขึ้นและมี GPU แรงกว่า สามารถเปลี่ยนเป็น
  `Qwen/Qwen2.5-3B-Instruct` หรือโมเดลไทยเฉพาะทางอย่าง `scb10x/llama3.1-typhoon2-8b-instruct` ได้
  (ตัวเลือกและวิธีสลับโมเดลอยู่ใน Cell การตั้งค่า)
- ใช้ **Chat Template** ของโมเดล (system = instruction, user = question, assistant = answer) เพื่อให้โมเดล
  เรียนรู้รูปแบบสนทนาแบบ Chatbot จริง ไม่ใช่แค่ต่อข้อความเปล่า ๆ
- วัดผลด้วย **chrF** ไม่ใช่ ROUGE เพราะ ROUGE ตัดคำด้วยช่องว่าง ส่วนภาษาไทยเขียนติดกันไม่มีช่องว่าง
  ทำให้คะแนน ROUGE ต่ำผิดปกติแม้โมเดลตอบถูก

## ⚠️ หมายเหตุสำคัญ

- Notebook นี้ **ต้องรันบน Kaggle/Colab ที่เปิดใช้ GPU** (Settings → Accelerator → GPU T4 x2 หรือ P100) และ
  ต้องมีอินเทอร์เน็ตเพื่อดาวน์โหลดโมเดลฐานจาก HuggingFace Hub
- ยังไม่มีชุด test แยก ผลที่วัดได้จาก validation จึงเป็นตัวเลขอ้างอิงคร่าว ๆ ถ้าต้องการวัดผลแบบเชื่อถือได้
  ควรแยกชุด test ออกมาต่างหากตั้งแต่ตอนเตรียมข้อมูล
- ทุกคำตอบในการเทรนมาจากข้อมูลจริงในไฟล์ JSON ต้นฉบับเท่านั้น ไม่มีการแต่งเติมข้อมูล

## 1. ติดตั้งไลบรารี

**ติดตั้งเฉพาะไลบรารีที่ยังไม่มี** (`peft`, `trl`, `bitsandbytes`) โดย **ไม่แตะ `transformers` /
`accelerate` / `numpy` / `pandas`** ที่ Colab ติดตั้งมาให้ตั้งแต่ต้น เพราะเวอร์ชันเหล่านั้นถูกจับคู่ให้เข้ากัน
(compatible) กันไว้แล้ว การไปสั่งเปลี่ยนเวอร์ชัน `transformers` มักทำให้ `numpy`/`pandas` ที่เหลือพังตามไปด้วย
(binary incompatibility)

ส่วน `torchao` ต้องถอนทิ้งเพราะ Colab มีเวอร์ชันเก่าติดมาซึ่งขัดกับ `peft` เวอร์ชันใหม่

In [ ]:
# ติดตั้งไลบรารีที่จำเป็น (peft, trl, bitsandbytes สำหรับ GPU)
# bitsandbytes ใช้ได้เฉพาะ CUDA/GPU เท่านั้น ถ้ารันบน TPU/CPU จะข้ามการใช้งาน 4bit อัตโนมัติ
# sacrebleu ใช้สำหรับคะแนน chrF ตอนประเมินผล
!pip install -q -U peft trl bitsandbytes accelerate evaluate sacrebleu
!pip uninstall -y torchao -q
print("ติดตั้งไลบรารีเสร็จสิ้น")

## 2. Import และตั้งค่าพื้นฐาน

In [ ]:
import os, json, glob, random
import torch
from datasets import Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ---------- ตรวจจับ Accelerator อัตโนมัติ (CUDA GPU / TPU-XLA / CPU) ----------
DEVICE_TYPE = "cpu"
IS_TPU = False
try:
    import torch_xla.core.xla_model as xm
    _ = xm.xla_device()
    IS_TPU = True
    DEVICE_TYPE = "tpu"
except Exception:
    if torch.cuda.is_available():
        DEVICE_TYPE = "cuda"
# ---------- ตั้งค่าโมเดลฐาน ----------
# ตัวเลือกอื่น: "Qwen/Qwen2.5-3B-Instruct" (คุณภาพสูงขึ้น ต้องการ GPU แรงกว่า)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# 4bit (bitsandbytes) ใช้ได้เฉพาะ CUDA GPU เท่านั้น -> เปิดอัตโนมัติเมื่อเป็น GPU
# ถ้าต้องการปิด 4bit บน GPU (เช่นใช้โมเดลเล็ก) ให้เปลี่ยนเป็น False
USE_4BIT = (DEVICE_TYPE == "cuda")
# bf16 ต้องใช้ GPU ตั้งแต่ Ampere (compute capability 8.0) ขึ้นไป เช่น A100 / L4
# T4 เป็น Turing (7.5) ไม่มี bf16 ในฮาร์ดแวร์ ถ้าฝืนใช้จะช้ากว่ามาก ต้องใช้ fp16 ถึงได้ tensor core เต็มที่
SUPPORTS_BF16 = IS_TPU or (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8)
COMPUTE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "lora_adapter_schedule_chatbot")

print("Accelerator ที่ตรวจพบ:", DEVICE_TYPE.upper())
print("ใช้ 4bit (bitsandbytes):", USE_4BIT)
print("Compute dtype:", COMPUTE_DTYPE, "| GPU รองรับ bf16:", SUPPORTS_BF16)
print("GPU พร้อมใช้งาน:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("อุปกรณ์ GPU:", torch.cuda.get_device_name(0), "| compute capability:", torch.cuda.get_device_capability())

## 3. โหลด Dataset (train / validation / test)

ค้นหาไฟล์ `.jsonl` จาก `/kaggle/input/...` ก่อน ถ้าไม่เจอ (เช่นรันในเครื่อง) จะค้นแบบ recursive จากโฟลเดอร์
ปัจจุบัน ทำให้เจอไฟล์ใน `dataset/` เองโดยไม่ต้องแก้ path

ไฟล์ test ไม่มีก็รันต่อได้ — ขั้นตอนทดสอบจะใช้ validation แทน และมีการตรวจ schema ของทุก record
(`instruction` / `question` / `answer`) ตั้งแต่ตอนโหลด เพื่อให้ error บอกตำแหน่งชัดแทนที่จะไปพังตอนเทรน

In [ ]:
def find_jsonl(keyword):
    # บน Kaggle ไฟล์อยู่ใต้ /kaggle/input, รันในเครื่องค้นแบบ recursive จากโฟลเดอร์ปัจจุบัน (เช่น dataset/)
    for pattern in (f"/kaggle/input/**/*{keyword}*.jsonl", f"**/*{keyword}*.jsonl"):
        matches = sorted(glob.glob(pattern, recursive=True))
        if matches:
            return matches[0]
    return None

train_path = find_jsonl("train")
val_path = find_jsonl("validation")
test_path = find_jsonl("test")

print("train.jsonl      :", train_path)
print("validation.jsonl :", val_path)
print("test.jsonl       :", test_path, "(ไม่มีก็ได้ — ขั้นทดสอบจะใช้ validation แทน)")

def load_jsonl(path):
    if path is None:
        return []
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_records = load_jsonl(train_path)
val_records = load_jsonl(val_path)
test_records = load_jsonl(test_path)

print(f"\nจำนวนข้อมูล Train      : {len(train_records)}")
print(f"จำนวนข้อมูล Validation : {len(val_records)}")
print(f"จำนวนข้อมูล Test       : {len(test_records)}")

assert len(train_records) > 0, "ไม่พบไฟล์ train.jsonl — ตรวจสอบว่าได้แนบไฟล์เป็น Input บน Kaggle แล้วหรือยัง"

# ตรวจ schema ตั้งแต่ตอนโหลด ถ้ามี record ขาด key จะ error พร้อมบอกตำแหน่ง
# แทนที่จะไปพังเป็น KeyError ลึก ๆ ตอน .map() หรือตอนเทรน
for split_name, records, required in (
    ("train", train_records, {"instruction", "question", "answer"}),
    ("validation", val_records, {"instruction", "question", "answer"}),
    ("test", test_records, {"question", "answer"}),
):
    bad = [i for i, r in enumerate(records) if not required.issubset(r)]
    assert not bad, f"{split_name}: record ที่ index {bad[:5]} ขาด key ที่ต้องมี {sorted(required)}"

print("\nตัวอย่างข้อมูล Train 1 รายการ:")
print(json.dumps(train_records[0], ensure_ascii=False, indent=2))

## 4. แปลงข้อมูลเป็นรูปแบบ Chat (system / user / assistant)

ใช้ `instruction` เป็น system prompt, `question` เป็นข้อความผู้ใช้, `answer` เป็นคำตอบของผู้ช่วย
แล้วแปลงเป็นข้อความเต็มด้วย Chat Template ของโมเดล เพื่อให้สอดคล้องกับรูปแบบที่โมเดลเข้าใจ

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# แปลงเป็นรูปแบบ prompt / completion เพื่อให้เทรน loss เฉพาะส่วนคำตอบ (completion_only_loss)
def to_prompt_completion(example):
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": example["question"]},
        
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    completion = example["answer"] + tokenizer.eos_token
    return {"prompt": prompt, "completion": completion}

train_dataset = Dataset.from_list(train_records).map(to_prompt_completion, remove_columns=list(train_records[0].keys()))
val_dataset = Dataset.from_list(val_records).map(to_prompt_completion, remove_columns=list(val_records[0].keys())) if val_records else None

print("ตัวอย่าง prompt:\n", train_dataset[0]["prompt"])
print("\nตัวอย่าง completion:\n", train_dataset[0]["completion"])

## 5. โหลดโมเดลฐาน (Base Model)

In [ ]:
# โหลดโมเดลฐาน โดยปรับ config อัตโนมัติตาม accelerator (GPU / TPU / CPU)
model_kwargs = {"torch_dtype": COMPUTE_DTYPE}
# device_map="auto" ใช้กับ GPU เท่านั้น (TPU/CPU ให้ปล่อยว่างแล้วย้าย device ทีหลัง)
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"

# เปิด 4bit เฉพาะเมื่อรันบน CUDA GPU (bitsandbytes ไม่รองรับ TPU/CPU)
if USE_4BIT:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
# บน TPU/CPU ที่ไม่ได้ใช้ device_map ให้ย้ายโมเดลไปยัง device เอง
if IS_TPU:
    import torch_xla.core.xla_model as xm
    model = model.to(xm.xla_device())

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

print("โหลดโมเดลฐานสำเร็จ:", MODEL_NAME)
print("จำนวนพารามิเตอร์ทั้งหมด:", sum(p.numel() for p in model.parameters()))

## 6. ตั้งค่า LoRA

เทรนเฉพาะพารามิเตอร์ Adapter ส่วนน้อย (rank 16) แทนการเทรนทั้งโมเดล เพื่อลดความเสี่ยง Overfitting
บน Dataset ขนาดเล็ก และประหยัดทรัพยากร

In [ ]:
# target_modules ครอบคลุมทั้ง attention (q,k,v,o) และ MLP (gate,up,down) เพื่อคุณภาพที่ดีขึ้น
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=TARGET_MODULES)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. ตั้งค่าการเทรน (Training Arguments) และสร้าง SFTTrainer

ข้อมูล train ~1,700 ตัวอย่าง (effective batch = 2 × 4 = 8) ตกราว 200 steps ต่อ epoch จึงใช้ **3 epoch**
คู่กับ **Early Stopping** ที่เฝ้าดู validation loss หากไม่ดีขึ้น 2 รอบติดจะหยุดให้เอง และโหลด checkpoint
ที่ดีที่สุดกลับมาตอนจบ (`load_best_model_at_end`)

⚠️ `trl` เปลี่ยน argument ของ `SFTConfig` บ่อยมากในแต่ละเวอร์ชัน (บางรุ่นตัด `max_seq_length` /
`warmup_ratio` ทิ้ง) โค้ดด้านล่างจึงตรวจสอบเองอัตโนมัติ 2 ชั้น:

1. **ชื่อ argument** — ส่งเฉพาะตัวที่ `SFTConfig` เวอร์ชันที่ติดตั้งจริงรองรับ
2. **ค่าของ argument** — ชื่อผ่านไม่ได้แปลว่าค่าผ่านด้วย (เช่น `loss_type="nll"` ที่ตัวเลือกต่างกันไป
   ตามเวอร์ชัน) จึงดักไว้ด้วย `try/except` แล้วลองใหม่โดยไม่ส่ง `loss_type` แทนที่จะพังทั้ง Cell

In [ ]:
import inspect
from transformers import EarlyStoppingCallback

USE_BF16 = (COMPUTE_DTYPE == torch.bfloat16)
USE_FP16 = (COMPUTE_DTYPE == torch.float16)
HAS_VAL = val_dataset is not None
print("USE_BF16 :", USE_BF16)
print("USE_FP16 :", USE_FP16)
print("HAS_VAL  :", HAS_VAL)

desired_sft_kwargs = dict(
    output_dir=os.path.join(OUTPUT_DIR, "sft_checkpoints"),
    # ---- Training ----
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=20,
    # ---- Evaluation / checkpoint ----
    eval_strategy="epoch" if HAS_VAL else "no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=HAS_VAL,   # โหลด checkpoint ที่ validation loss ดีที่สุด
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    # bf16 บน Ampere ขึ้นไป, fp16 บน Turing (T4) — ต้องเปิดอย่างใดอย่างหนึ่งเท่านั้น
    bf16=USE_BF16,
    fp16=USE_FP16,
    report_to=[],                      # ไม่ส่งผลการเทรนไป WandB / TensorBoard
    # ---- SFT ----
    max_length=768,
    packing=False,
    completion_only_loss=True,         # คิด loss เฉพาะส่วนคำตอบ
    # กัน TRL รุ่นใหม่ patch chunked CE ซึ่งชนกับ model.forward ที่เป็น functools.partial
    # (AttributeError: 'functools.partial' object has no attribute '__func__')
    loss_type="nll",
    seed=RANDOM_SEED,
)

# TRL เปลี่ยน argument ของ SFTConfig บ่อย -> ส่งเฉพาะตัวที่เวอร์ชันที่ติดตั้งจริงรองรับ
valid_params = set(inspect.signature(SFTConfig.__init__).parameters)
sft_kwargs = {k: v for k, v in desired_sft_kwargs.items() if k in valid_params}
dropped = set(desired_sft_kwargs) - set(sft_kwargs)
if dropped:
    print("SFTConfig เวอร์ชันนี้ไม่รองรับ argument:", dropped, "-> ข้ามไป")

# ชื่อ argument ผ่านแล้ว แต่ "ค่า" อาจยังไม่ผ่าน (choices ของ loss_type ต่างกันในแต่ละเวอร์ชัน)
try:
    sft_config = SFTConfig(**sft_kwargs)
except (ValueError, TypeError) as e:
    if "loss_type" not in sft_kwargs:
        raise
    print("SFTConfig ไม่รับ loss_type='nll' (", e, ") -> ลองใหม่โดยไม่ส่ง loss_type")
    sft_kwargs.pop("loss_type")
    sft_config = SFTConfig(**sft_kwargs)

print("\nสร้าง SFTConfig สำเร็จ")
print("loss_type:", getattr(sft_config, "loss_type", "ไม่พบในเวอร์ชันนี้"))

trainer_kwargs = dict(model=model, args=sft_config, train_dataset=train_dataset)

# ถ้ามี validation: เปิด eval + Early Stopping (val loss ไม่ดีขึ้น 2 รอบติด = หยุด กัน Overfitting)
if HAS_VAL:
    trainer_kwargs["eval_dataset"] = val_dataset
    trainer_kwargs["callbacks"] = [EarlyStoppingCallback(early_stopping_patience=2)]
    print("เปิดใช้ Early Stopping: patience=2")

# TRL รุ่นใหม่ใช้ processing_class, รุ่นเก่าใช้ tokenizer
trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters)
tokenizer_arg = next((a for a in ("processing_class", "tokenizer") if a in trainer_params), None)
if tokenizer_arg:
    trainer_kwargs[tokenizer_arg] = tokenizer
    print("ใช้ tokenizer argument:", tokenizer_arg)
else:
    print("คำเตือน: SFTTrainer เวอร์ชันนี้ไม่พบ processing_class หรือ tokenizer")

trainer = SFTTrainer(**trainer_kwargs)

print("\n" + "=" * 60)
print("สร้าง SFTTrainer สำเร็จ — พร้อมเริ่มเทรน")
print("=" * 60)

## 8. เริ่ม Fine-tuning

In [ ]:
train_result = trainer.train()

print("\nเทรนเสร็จสิ้น")
print("Train loss สุดท้าย:", train_result.training_loss)


## 9. ประเมินผลบน Validation Set

In [ ]:
if val_dataset is not None:
    eval_result = trainer.evaluate()
    print("ผลการประเมินบน Validation set:")
    for k, v in eval_result.items():
        print(f"  {k}: {v}")
else:
    print("ไม่มี validation.jsonl ให้ข้ามขั้นตอนนี้")


## 10. บันทึก LoRA Adapter

In [ ]:
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("บันทึก Adapter ไว้ที่:", ADAPTER_DIR)
print("ไฟล์ที่ได้:")
for fn in os.listdir(ADAPTER_DIR):
    print(" -", fn)


## 11. ทดสอบตอบคำถามจริง (Qualitative Check)

ถามคำถามจากชุด test ถ้ามี ถ้าไม่มีจะใช้ validation แทน (โมเดลไม่ได้เทรนบนทั้งสองชุด) แล้วเทียบคำตอบของโมเดล
กับคำตอบอ้างอิง

วัดด้วย **chrF** (เทียบ n-gram ระดับตัวอักษร 0-100 สูง = ดี) และ **exact match** — ไม่ใช้ ROUGE เพราะ ROUGE
ตัดคำด้วยช่องว่าง ซึ่งภาษาไทยไม่มี ทำให้คะแนนต่ำผิดปกติแม้คำตอบถูกต้อง

In [ ]:
import evaluate

# เปิด use_cache กลับมาเพื่อให้ generate เร็วขึ้น (ตอนเทรนปิดไว้เพราะ gradient checkpointing)
model.config.use_cache = True
model.eval()

DEFAULT_INSTRUCTION = "ตอบคำถามเกี่ยวกับตารางสอนของอาจารย์"

def generate_answer(question, instruction=DEFAULT_INSTRUCTION, max_new_tokens=256):
    messages = [{"role": "system", "content": instruction}, {"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

# ----- Qualitative check: เทียบคำตอบโมเดลกับคำตอบอ้างอิง -----
# ถ้าไม่มี test.jsonl ใช้ validation แทนได้ (โมเดลไม่ได้เทรนบน validation เหมือนกัน)
sample_records = test_records or val_records
if not sample_records:
    print("ไม่พบทั้ง test และ validation — ข้ามการประเมินคุณภาพคำตอบ")
else:
    print(f"ประเมินจากชุด: {'test' if test_records else 'validation'}")
    sample_records = sample_records[:20]
    preds, refs = [], []
    for rec in sample_records:
        preds.append(generate_answer(rec["question"], rec.get("instruction", DEFAULT_INSTRUCTION)))
        refs.append(rec["answer"])

    # ใช้ chrF ไม่ใช่ ROUGE เพราะ ROUGE ตัดคำด้วยช่องว่าง ซึ่งภาษาไทยไม่มี -> คะแนนเพี้ยนเกือบ 0
    # ทั้งที่ตอบถูก ส่วน chrF เทียบ n-gram ระดับตัวอักษร จึงใช้กับภาษาไทยได้ตรงกว่า
    chrf = evaluate.load("chrf").compute(predictions=preds, references=[[r] for r in refs])
    exact_match = sum(p.strip() == r.strip() for p, r in zip(preds, refs)) / len(preds)

    print(f"\n===== คะแนนคุณภาพคำตอบ (n={len(preds)}) =====")
    print(f"  chrF (0-100, สูง = ดี) : {chrf['score']:.2f}")
    print(f"  ตอบตรงเป๊ะ (exact match) : {exact_match:.1%}")

    # ดูเฉพาะเคสที่ตอบไม่ตรง เพื่อแยกว่าพลาดแค่รายละเอียด หรือผิดข้อเท็จจริง
    wrong = [(r, p) for r, p in zip(sample_records, preds) if p.strip() != r["answer"].strip()]
    print(f"\n===== ตอบไม่ตรง {len(wrong)}/{len(preds)} รายการ =====")
    for rec, pred in wrong:
        print("คำถาม       :", rec["question"])
        print("คำตอบอ้างอิง :", rec["answer"])
        print("คำตอบโมเดล   :", pred)
        print("-" * 60)

## 12. (ทางเลือก) รวม LoRA Adapter เข้ากับโมเดลฐาน สำหรับใช้งาน Deploy

หากต้องการโมเดลไฟล์เดียวสมบูรณ์ (ไม่ต้องพก Adapter แยก) สามารถ merge เข้ากับ Base Model ได้
*(ใช้ RAM/VRAM มากขึ้นชั่วขณะตอน merge)*

In [ ]:
MERGE_MODEL = False  # เปลี่ยนเป็น True หากต้องการรวมโมเดลจริง

if not MERGE_MODEL:
    print("ข้ามขั้นตอนการ merge (ตั้งค่า MERGE_MODEL = True เพื่อเปิดใช้งาน)")
elif USE_4BIT:
    # merge adapter เข้ากับ base ที่ถูก quantize 4bit ไว้ไม่ได้ ต้องโหลด base เป็น bf16/fp16 ใหม่ก่อน
    raise RuntimeError(
        "merge ไม่รองรับโมเดลที่โหลดแบบ 4bit — ตั้ง USE_4BIT = False ใน Cell ตั้งค่า แล้วรันใหม่ตั้งแต่ Cell โหลดโมเดลฐาน"
    )
else:
    merged_model = trainer.model.merge_and_unload()
    merged_dir = os.path.join(OUTPUT_DIR, "merged_schedule_chatbot_model")
    merged_model.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)
    print("บันทึกโมเดลที่ merge แล้วไว้ที่:", merged_dir)

## 13. สรุปผล

In [ ]:
print("=" * 60)
print("สรุปการ Fine-tune Chatbot ตารางสอนอาจารย์")
print("=" * 60)
print(f"โมเดลฐาน           : {MODEL_NAME}")
print(f"เทคนิค              : LoRA (r=16, alpha=32)")
print(f"จำนวนข้อมูล Train   : {len(train_records)}")
print(f"จำนวนข้อมูล Val     : {len(val_records)}")
print(f"จำนวนข้อมูล Test    : {len(test_records)}")
print(f"Train loss สุดท้าย  : {train_result.training_loss:.4f}")
print(f"Adapter บันทึกที่   : {ADAPTER_DIR}")
print()
if not test_records:
    print("⚠️ ยังไม่มีชุด test แยก — ผลที่วัดได้มาจาก validation จึงเป็นตัวเลขอ้างอิงคร่าว ๆ")
print(f"⚠️ Dataset ขนาด {len(train_records) + len(val_records)} คู่ และเนื้อหาซ้ำแนวกันสูง เหมาะกับการใช้เป็นต้นแบบ")
print("หากต้องการ Chatbot ที่แม่นยำและครอบคลุมกว่านี้ ควรเพิ่มความหลากหลายของคำถามก่อน Fine-tune รอบถัดไป")

In [ ]:
# บีบอัด Adapter เป็น .zip สำหรับดาวน์โหลด (ใช้ตัวเดียวกับที่บันทึกไว้ใน Cell 10)
!zip -r -q {ADAPTER_DIR}.zip {ADAPTER_DIR}
print("ไฟล์ zip:", ADAPTER_DIR + ".zip")